# Prediction of Annual Return from Fundamental Data and Market Cap
Chapter 4 of the book: "Build Your Own AI Investor"

For our investing AI to select stocks for investment it will need to be able to predict which stocks are likely to go up. WIth our X and y data we can train any of the machine learning algorithms to do this. We'll try using all of them and keep the ones that show promise.

In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import gaussian_kde

from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    ShuffleSplit,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree

In [ ]:
plt.rcParams["figure.figsize"] = [7.0, 4.5]
plt.rcParams["figure.dpi"] = 150

## Configuration

In [ ]:
# --- File paths ---------------------------------------------------------------
RATIOS_FILE = "Annual_Stock_Price_Fundamentals_Ratios.csv"
PERF_FILE = "Annual_Stock_Price_Performance_Percentage.csv"
MODEL_DIR = Path(".")  # directory for saved model files

# --- Experiment settings ------------------------------------------------------
TEST_SIZE = 0.1
RANDOM_STATE = 42
N_EVAL_RUNS = 10        # number of train/test splits for portfolio evaluation
TOP_N = 10              # number of stocks in top/bottom portfolio slices

In [ ]:
def load_data(
    ratios_path: str = RATIOS_FILE,
    perf_path: str = PERF_FILE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.Series]:
    """Load X (ratios) and y (performance), shuffle rows deterministically."""
    X = pd.read_csv(ratios_path, index_col=0)
    y = pd.read_csv(perf_path, index_col=0)["Perf"]

    # Shuffle together so X and y stay aligned
    shuffled_idx = X.sample(frac=1.0, random_state=random_state).index
    return X.loc[shuffled_idx], y.loc[shuffled_idx]

In [ ]:
X, y = load_data()
print(f"X: {X.shape}  |  y: {y.shape}  |  mean return: {y.mean():.4f}")

## Helper Functions
Reusable utilities for model training, evaluation, and plotting.

In [ ]:
def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Standard train/test split with summary printout."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=random_state,
    )
    print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
    return X_train, X_test, y_train, y_test


def fit_and_evaluate(
    pipeline,
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    *,
    save_as: str | None = None,
) -> np.ndarray:
    """Fit a pipeline, print train/test MSE, optionally save model. Returns y_pred."""
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    train_mse = mean_squared_error(y_train, pipeline.predict(X_train))
    test_mse = mean_squared_error(y_test, y_pred)
    print(f"Train MSE: {train_mse:.6f}  |  Test MSE: {test_mse:.6f}")
    if save_as:
        path = MODEL_DIR / save_as
        pickle.dump(pipeline, open(path, "wb"))
        print(f"Model saved to {path}")
    return y_pred


def print_top_bottom(y_test: pd.Series, y_pred: np.ndarray, n: int = TOP_N) -> None:
    """Print mean predicted/actual returns for top-N and bottom-N predicted stocks."""
    results = (
        pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred})
        .sort_values("Predicted", ascending=False)
        .reset_index(drop=True)
    )
    for label, slc in [("Top", slice(None, n)), ("Bottom", slice(-n, None))]:
        pred_mean = results["Predicted"].iloc[slc].mean()
        actual_mean = results["Actual"].iloc[slc].mean()
        print(f"{label} {n}  —  Predicted: {pred_mean:.4f}  |  Actual: {actual_mean:.4f}")

In [ ]:
def evaluate_portfolio_picks(
    pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    *,
    n_runs: int = N_EVAL_RUNS,
    n_stocks: int = TOP_N,
    verbose: bool = True,
) -> tuple[list[float], list[float]]:
    """Run *n_runs* train/test splits, collect top/bottom-N actual returns.

    Returns (top_actual_returns, bottom_actual_returns) as lists of
    mean % return per run.
    """
    top_pred, top_actual = [], []
    bot_pred, bot_actual = [], []

    for i in range(n_runs):
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE + i,
        )
        pipeline.fit(Xtr, ytr)
        yp = pipeline.predict(Xte)

        results = (
            pd.DataFrame({"Actual": yte.values, "Predicted": yp})
            .sort_values("Predicted", ascending=False)
            .reset_index(drop=True)
        )
        top_pred.append(round(results["Predicted"].iloc[:n_stocks].mean() * 100, 2))
        top_actual.append(round(results["Actual"].iloc[:n_stocks].mean() * 100, 2))
        bot_pred.append(round(results["Predicted"].iloc[-n_stocks:].mean() * 100, 2))
        bot_actual.append(round(results["Actual"].iloc[-n_stocks:].mean() * 100, 2))

    if verbose:
        _arr = lambda lst: np.array(lst)  # noqa: E731
        print(f"Top {n_stocks} predicted returns : {top_pred}")
        print(f"Top {n_stocks} actual returns    : {top_actual}\n")
        print(f"Bottom {n_stocks} predicted returns: {bot_pred}")
        print(f"Bottom {n_stocks} actual returns   : {bot_actual}")
        print("─" * 50)
        print(f"Mean top predicted : {_arr(top_pred).mean():.2f}  (σ {_arr(top_pred).std():.2f})")
        print(f"Mean top actual    : {_arr(top_actual).mean():.2f}  (σ {_arr(top_actual).std():.2f})")
        print(f"Mean bottom predicted: {_arr(bot_pred).mean():.2f}  (σ {_arr(bot_pred).std():.2f})")
        print(f"Mean bottom actual   : {_arr(bot_actual).mean():.2f}  (σ {_arr(bot_actual).std():.2f})")
        print("─" * 50)

    return top_actual, bot_actual

In [ ]:
def plot_pred_vs_actual(
    model_name: str,
    y_pred: np.ndarray,
    y_actual: np.ndarray,
    limit: float = 2.0,
    *,
    kde_contour: bool = False,
    ax: plt.Axes | None = None,
) -> None:
    """Scatter plot of predicted vs actual returns with a linear fit line.

    If *kde_contour* is True, overlay a kernel-density contour.
    """
    ax = ax or plt.gca()

    if kde_contour:
        k = gaussian_kde(np.stack([y_pred, y_actual]))
        xi, yi = np.mgrid[-limit:limit:160j, -limit:limit:160j]
        zi = k(np.vstack([xi.flatten(), yi.flatten()]))
        ax.pcolormesh(xi, yi, zi.reshape(xi.shape), shading="gouraud", cmap=plt.cm.Greens)
        ax.contour(xi, yi, zi.reshape(xi.shape))

    ax.scatter(y_pred, y_actual, s=1, alpha=0.5)

    # Linear fit (swap axes because predictions cluster around a narrow band)
    fit = LinearRegression().fit(y_actual.reshape(-1, 1), y_pred.reshape(-1, 1))
    xx = np.array([[-limit], [limit]])
    ax.plot(fit.predict(xx), xx, "g", label="Linear fit")
    ax.plot([-100, 100], [-100, 100], "y--", label="y = x (perfect)")
    ax.axhline(0, color="r", lw=0.5)
    ax.axvline(0, color="r", lw=0.5)
    ax.set(
        xlabel="Predicted Return",
        ylabel="Actual Return",
        xlim=(-limit * 0.2, limit * 0.6) if kde_contour else (-limit, limit),
        ylim=(-limit, limit),
        title=model_name,
    )
    ax.legend(fontsize=8)
    ax.grid(True)


def plot_top_bottom_scatter(
    y_pred: np.ndarray,
    y_test: pd.Series,
    n: int = 100,
    ylim: tuple[float, float] = (-1, 7),
) -> None:
    """Side-by-side scatter of the best and worst N predicted stocks."""
    results = (
        pd.DataFrame({"Predicted": y_pred, "Actual": y_test.values})
        .sort_values("Predicted")
        .reset_index(drop=True)
    )
    fig, (ax_low, ax_high) = plt.subplots(1, 2, figsize=(14, 7))
    for ax, slc, title in [
        (ax_low, slice(None, n), f"Bottom {n} predictions"),
        (ax_high, slice(-n, None), f"Top {n} predictions"),
    ]:
        chunk = results.iloc[slc]
        ax.plot(chunk.index, chunk["Predicted"], "x", label="Predicted")
        ax.plot(chunk.index, chunk["Actual"], "x", label="Actual")
        ax.set(ylim=ylim, ylabel="Annual Return", xlabel="Rank (low → high predicted)")
        ax.set_title(title)
        ax.legend()
        ax.grid(True)
    fig.tight_layout()

In [ ]:
def plot_learning_curve(
    estimator,
    X: pd.DataFrame,
    y: pd.Series,
    *,
    train_sizes: list[float] | None = None,
    title: str = "Learning Curve",
    scoring: str = "neg_mean_squared_error",
    cv=None,
    n_jobs: int = 4,
) -> pd.DataFrame:
    """Compute and plot a learning curve, returning the results DataFrame."""
    if train_sizes is None:
        train_sizes = [0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
    if cv is None:
        cv = ShuffleSplit(n_splits=100, test_size=0.2, random_state=RANDOM_STATE)

    sizes, train_scores, test_scores, fit_times, _ = learning_curve(
        estimator, X, y,
        cv=cv, scoring=scoring, n_jobs=n_jobs,
        train_sizes=train_sizes, return_times=True,
    )
    results = pd.DataFrame(
        {
            "train_rmse": np.sqrt(-train_scores.mean(axis=1)),
            "train_std": np.sqrt(-train_scores).std(axis=1),
            "test_rmse": np.sqrt(-test_scores.mean(axis=1)),
            "test_std": np.sqrt(-test_scores).std(axis=1),
            "fit_time": fit_times.mean(axis=1),
        },
        index=sizes,
    )
    ax = results[["train_rmse", "test_rmse"]].plot(style="-x")
    ax.fill_between(sizes, results["train_rmse"] - results["train_std"],
                     results["train_rmse"] + results["train_std"], alpha=0.2)
    ax.fill_between(sizes, results["test_rmse"] - results["test_std"],
                     results["test_rmse"] + results["test_std"], alpha=0.2)
    ax.set(xlabel="Training rows", ylabel="RMSE", title=title)
    ax.legend(["Train RMSE", "Test RMSE"])
    ax.grid(True)
    return results

# Linear Regression
As a start try vanilla linear regression to get the ball rolling.
We use the powertransformer in a pipeline with our linear regressor.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
pl_linear = Pipeline([
    ("power", PowerTransformer()),
    ("linear", LinearRegression()),
])
y_pred = fit_and_evaluate(pl_linear, X_train, X_test, y_train, y_test, save_as="pl_linear.p")

The Errors aren't that good, and they can vary a lot depending on train/test split. Let's try many runs and see if it the regressor is actually learning anything.

In [ ]:
lc_results = plot_learning_curve(pl_linear, X, y, title="Linear Regression Learning Curve")

In [ ]:
lc_results

### Linear Regression Prediction Analysis
Let's see how good our predictions are in a bit more depth. Our AI will be depending on these predictions so we need to be sure stocks can be picked well in this chapter.

In [ ]:
plot_top_bottom_scatter(y_pred, y_test)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_pred_vs_actual("Linear Regression", y_pred, y_test.to_numpy(), ax=ax)

It doesn't look so good visually, but there seems to be some ability there. Let's take a closer look by taking our predictors top few stock return predictions and comparing them to reality.

In [ ]:
print_top_bottom(y_test, y_pred)

There is *some* predictive ability here, it is definately worthwhile using linear regression in the backtest under greater scrutiny later. Don't worry about survivorship bias at this point, we can account for that later, besides these predicted returns are high enough to compensate for that if you look at bankruptcy statistics, and the kinds of stocks being chosen.

Let's try the model with a few more train/test samplings by changing the random_state, and see if the predictive ability stays.

In [ ]:
evaluate_portfolio_picks(pl_linear, X, y)

# Elastic Net Regression
Let's see if some regularisation form linear regression will get us better results.

In [ ]:
pl_elastic_net = Pipeline([
    ("power", PowerTransformer()),
    ("elastic_net", ElasticNet()),
])
y_pred_en = fit_and_evaluate(
    pl_elastic_net, X_train, X_test, y_train, y_test, save_as="pl_ElasticNet.p",
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_pred_vs_actual("ElasticNet (default)", y_pred_en, y_test.to_numpy(), ax=ax)

### Test some Hyperparameters to try and improve prediction

In [ ]:
# Try with low L1 ratio
pl_elastic_net_low_l1 = Pipeline([
    ("power", PowerTransformer()),
    ("elastic_net", ElasticNet(l1_ratio=0.00001)),
])
y_pred_en_low = fit_and_evaluate(pl_elastic_net_low_l1, X_train, X_test, y_train, y_test)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_pred_vs_actual("ElasticNet (default)", y_pred_en, y_test.to_numpy(), ax=ax1)
plot_pred_vs_actual("ElasticNet (L1=0.00001)", y_pred_en_low, y_test.to_numpy(), ax=ax2)
fig.tight_layout()

In [ ]:
print_top_bottom(y_test, y_pred_en)

### See if the results are repeatable.

In [ ]:
evaluate_portfolio_picks(pl_elastic_net, X, y)

# K Nearest Neighbours Regression

In [ ]:
pl_knn = Pipeline([
    ("power", PowerTransformer()),
    ("knn", KNeighborsRegressor(n_neighbors=40)),
])
y_pred_knn = fit_and_evaluate(pl_knn, X_train, X_test, y_train, y_test, save_as="pl_KNeighbors.p")

Again the results are fickle depending on the train/test split. Better to see statistical results.

### How does performance change with K? Take a look at Validation curve
Learning curve->See errors as you change the number of training rows

Validation curve->See errors as you change one of the model parameters

In [ ]:
def knn_validation_curve(
    X: pd.DataFrame,
    y: pd.Series,
    k_values: list[int] | None = None,
    n_runs: int = 40,
) -> pd.DataFrame:
    """Compute train/test MSE across different K values with multiple splits."""
    k_values = k_values or [4, 8, 16, 32, 64, 100]
    records = []
    for k in k_values:
        print(f"  K={k}", end="")
        for j in range(n_runs):
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE + j)
            pipe = Pipeline([("power", PowerTransformer()), ("knn", KNeighborsRegressor(n_neighbors=k))])
            pipe.fit(Xtr, ytr)
            records.append({
                "K": k,
                "train_mse": mean_squared_error(ytr, pipe.predict(Xtr)),
                "test_mse": mean_squared_error(yte, pipe.predict(Xte)),
            })
    print()
    return pd.DataFrame(records)

knn_val = knn_validation_curve(X, y)

In [ ]:
knn_val.head()

In [ ]:
knn_summary = knn_val.groupby("K").agg(["mean", "std"])
knn_summary.columns = ["_".join(c) for c in knn_summary.columns]
knn_summary

In [ ]:
fig, ax = plt.subplots()
for metric, label in [("train_mse_mean", "Train MSE"), ("test_mse_mean", "Test MSE")]:
    std_col = metric.replace("mean", "std")
    ax.plot(knn_summary.index, knn_summary[metric], "-x", label=label)
    ax.fill_between(
        knn_summary.index,
        knn_summary[metric] - knn_summary[std_col],
        knn_summary[metric] + knn_summary[std_col],
        alpha=0.2,
    )
ax.set(xlabel="K", ylabel="MSE", title="K-NN Validation Curve")
ax.legend()
ax.grid(True)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_pred_vs_actual("K-NN Regressor", y_pred_knn, y_test.to_numpy(), ax=ax)

In [ ]:
print_top_bottom(y_test, y_pred_knn)

In [ ]:
evaluate_portfolio_picks(pl_knn, X, y)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_pred_vs_actual("K-NN (KDE)", y_pred_knn, y_test.to_numpy(), limit=1.0, kde_contour=True, ax=ax)

# Support Vector Machine Regression
Warning: From grid search onward this takes a VERY long time and doesn't do that well relative to training time.
### Quick SVM Regressor with default parameters

In [ ]:
pl_svm = Pipeline([
    ("power", PowerTransformer()),
    ("svr", SVR()),  # default hyperparameters
])
y_pred_svm = fit_and_evaluate(pl_svm, X_train, X_test, y_train, y_test, save_as="pl_svm.p")

In [ ]:
evaluate_portfolio_picks(pl_svm, X, y)

### As there are many parameters do a GridSearch CV, to find optimal parameters
We aren't actually aiming for prediction ACCURACY though, we want stock return. Skip this part in the book.

In [ ]:
# Grid search (takes a VERY long time — uncomment to run)
# Best found: C=1, gamma=0.001, epsilon=0.2
#
# from sklearn.model_selection import GridSearchCV
# params = [
#     {"svr__kernel": ["linear"], "svr__C": [1, 10, 100], "svr__epsilon": [0.05, 0.1, 0.2]},
#     {"svr__kernel": ["rbf"], "svr__C": [1, 10, 100], "svr__gamma": [0.001, 0.01, 0.1], "svr__epsilon": [0.05, 0.1, 0.2]},
# ]
# gs = GridSearchCV(pl_svm, params, cv=10, scoring="neg_mean_squared_error", n_jobs=4)
# gs.fit(X_train, y_train)
# print(gs.best_params_)

In [ ]:
# SVM with grid-search-optimal parameters
pl_svm_opt = Pipeline([
    ("power", PowerTransformer()),
    ("svr", SVR(kernel="rbf", C=1, gamma=0.001, epsilon=0.2)),
])
y_pred_svm_opt = fit_and_evaluate(
    pl_svm_opt, X_train, X_test, y_train, y_test, save_as="pl_svm.p",
)

fig, ax = plt.subplots(figsize=(6, 6))
plot_pred_vs_actual("SVR (optimal)", y_pred_svm_opt, y_test.to_numpy(), ax=ax)

In [ ]:
evaluate_portfolio_picks(pl_svm_opt, X, y)

### Optimal prediction ability may not be optimal for US (Own selected parameters)
yes the prediction coulld be more accurate, but if it is producing lower returns for us then the MSE measure of error isn't really what we want, even if it might tend towards what we want.

In [ ]:
# Hyperparameter sweep for best stock-return (not MSE) — uncomment to run
# Takes a long time
# for kern, c_val, eps_val, gam_val in itertools.product(
#     ["linear", "rbf"], [1, 10, 100], [0.05, 0.1, 0.2], [0.001, 0.01, 0.1]
# ):
#     params = dict(kernel=kern, C=c_val, epsilon=eps_val)
#     if kern == "rbf":
#         params["gamma"] = gam_val
#     pipe = Pipeline([("power", PowerTransformer()), ("svr", SVR(**params))])
#     top, _ = evaluate_portfolio_picks(pipe, X, y, verbose=False)
#     print(f"return: {np.mean(top):.2f}  σ {np.std(top):.2f}  | {params}")

In [ ]:
# SVM that maximises stock returns (not prediction accuracy)
pl_svm_returns = Pipeline([
    ("power", PowerTransformer()),
    ("svr", SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.1)),
])
y_pred_svm_ret = fit_and_evaluate(
    pl_svm_returns, X_train, X_test, y_train, y_test, save_as="pl_svm.p",
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_pred_vs_actual("SVR (return-optimised)", y_pred_svm_ret, y_test.to_numpy(), ax=ax)

In [ ]:
print_top_bottom(y_test, y_pred_svm_ret)

In [ ]:
evaluate_portfolio_picks(pl_svm_returns, X, y)

# Decision Tree Regression

In [ ]:
pl_dec_tree = Pipeline([
    ("tree", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])
y_pred_tree = fit_and_evaluate(pl_dec_tree, X_train, X_test, y_train, y_test, save_as="pl_decTree.p")

That train error looks VERY low, it's likely overfitted. let's create a learning curve and find a good value for max_depth

In [ ]:
depths = range(2, 50)
train_errs, test_errs = [], []
for d in depths:
    pipe = Pipeline([("tree", DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=d))])
    pipe.fit(X_train, y_train)
    train_errs.append(mean_squared_error(y_train, pipe.predict(X_train)))
    test_errs.append(mean_squared_error(y_test, pipe.predict(X_test)))

fig, ax = plt.subplots()
ax.plot(list(depths), np.sqrt(train_errs), "r", label="Train RMSE")
ax.plot(list(depths), np.sqrt(test_errs), "b", label="Test RMSE")
ax.set(xlabel="max_depth", ylabel="RMSE")
ax.legend()
ax.grid(True)

We can't just choose a max_depth of 40+ because it is obviously overfitting there. The error vs. the training data is unfortunately not coming towards the error vs. testing data, so a lower tree depth is desirable, bt we can't choose a very low number, remember there are 15 columns to our x matrix (and possibly more if you have generated your own ratios).

We'll settle on a tree_depth of 10. Any less and we will obviously be utilising too little of our data, yet the error between test and training sets is reasonably close.

### K-Fold cross validation
We'll want to check how repeatable the decision tree regressor is. 

In [ ]:
pipe_cv = Pipeline([("tree", DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=10))])
scores = cross_validate(pipe_cv, X, y, scoring="neg_mean_squared_error", cv=10, return_train_score=True)

train_rmse = np.sqrt(-scores["train_score"])
test_rmse = np.sqrt(-scores["test_score"])
print(f"Train RMSE: {train_rmse.round(2)}  →  mean {train_rmse.mean():.4f} (σ {train_rmse.std():.4f})")
print(f"Test  RMSE: {test_rmse.round(2)}  →  mean {test_rmse.mean():.4f} (σ {test_rmse.std():.4f})")

Seems consistent, see if returns are consistent.

### Investigate Predictive Ability
What max tree depth shall we use?

In [ ]:
for depth in [4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 50]:
    pipe = Pipeline([("tree", DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=depth))])
    top, _ = evaluate_portfolio_picks(pipe, X, y, verbose=False)
    print(f"max_depth={depth:>3d}  →  mean return: {np.mean(top):.2f}%  (σ {np.std(top):.2f})")

### Use max_depth of 15

In [ ]:
pl_dec_tree = Pipeline([
    ("tree", DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=15)),
])
y_pred_tree = fit_and_evaluate(
    pl_dec_tree, X_train, X_test, y_train, y_test, save_as="pl_decTree.p",
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_pred_vs_actual("Decision Tree (depth=15)", y_pred_tree, y_test.to_numpy(), ax=ax)

In [ ]:
print_top_bottom(y_test, y_pred_tree)

In [ ]:
evaluate_portfolio_picks(pl_dec_tree, X, y)

In [ ]:
# Visualise the full-depth tree (truncated to depth 2 for readability)
deep_tree = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=40)
deep_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(10, 10), dpi=400)
plot_tree(deep_tree, feature_names=X.columns, filled=True, max_depth=2, fontsize=10, ax=ax)
fig.savefig("RegDecTree.png")

# Random Forest Regression

In [ ]:
for depth in range(4, 21):
    rf = RandomForestRegressor(random_state=RANDOM_STATE, max_depth=depth)
    top, _ = evaluate_portfolio_picks(rf, X, y, verbose=False)
    print(f"max_depth={depth:>2d}  →  mean return: {np.mean(top):.2f}%  (σ {np.std(top):.2f})")

In [ ]:
rf_regressor = RandomForestRegressor(random_state=RANDOM_STATE, max_depth=10)
y_pred_rf = fit_and_evaluate(rf_regressor, X_train, X_test, y_train, y_test, save_as="pl_rfregressor.p")

In [ ]:
et_regressor = ExtraTreesRegressor(random_state=RANDOM_STATE, max_depth=10)
y_pred_et = fit_and_evaluate(et_regressor, X_train, X_test, y_train, y_test, save_as="pl_ETregressor.p")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_pred_vs_actual("Random Forest", y_pred_rf, y_test.to_numpy(), ax=ax1)
plot_pred_vs_actual("Extra Trees", y_pred_et, y_test.to_numpy(), ax=ax2)
fig.tight_layout()

In [ ]:
evaluate_portfolio_picks(rf_regressor, X, y)

In [ ]:
evaluate_portfolio_picks(et_regressor, X, y)

In [ ]:
importance = pd.Series(rf_regressor.feature_importances_, index=X.columns).sort_values()
importance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
importance.plot.barh(ax=ax)
ax.set_title("Random Forest Feature Importance", fontsize=15)
ax.grid(True)

In [ ]:
# Re-fit for scatter analysis
rf_regressor.fit(X_train, y_train)
y_pred_rf = rf_regressor.predict(X_test)

results_rf = (
    pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred_rf})
    .sort_values("Predicted", ascending=False)
    .reset_index(drop=True)
)

n_show = 100
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
for ax, slc, title in [
    (ax1, slice(None, n_show), f"Top {n_show} Predicted"),
    (ax2, slice(-n_show, None), f"Bottom {n_show} Predicted"),
]:
    chunk = results_rf.iloc[slc]
    ax.scatter(chunk.index, chunk["Actual"], s=30)
    ax.set(ylim=(-1, 5), ylabel="Stock Return", xlabel="Rank (highest predicted → left)")
    ax.set_title(f"{title} — Actual Returns", fontsize=16)
    ax.grid(True)
fig.tight_layout()

# Gradient Boosted Decision Tree Regressor

In [ ]:
pl_grad_boost = Pipeline([
    ("gb", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=10,
        random_state=RANDOM_STATE,
        loss="squared_error",
    )),
])
y_pred_gb = fit_and_evaluate(
    pl_grad_boost, X_train, X_test, y_train, y_test, save_as="pl_GradBregressor.p",
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_pred_vs_actual("Gradient Boosting", y_pred_gb, y_test.to_numpy(), ax=ax)

In [ ]:
print_top_bottom(y_test, y_pred_gb)

In [ ]:
evaluate_portfolio_picks(pl_grad_boost, X, y)

# Plot all results

In [ ]:
# Collect portfolio picks from all models
MODEL_REGISTRY: dict[str, object] = {
    "LinearRegr": pl_linear,
    "ElasticNet": pl_elastic_net,
    "KNN": pl_knn,
    "SVR": pl_svm_returns,
    "DecTree": pl_dec_tree,
    "RF": rf_regressor,
    "ExtraTrees": et_regressor,
    "GradBoost": pl_grad_boost,
}

df_best, df_worst = pd.DataFrame(), pd.DataFrame()
for name, model in MODEL_REGISTRY.items():
    print(f"Evaluating {name}...")
    top, bot = evaluate_portfolio_picks(model, X, y, verbose=False)
    df_best[name] = top
    df_worst[name] = bot

print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
df_best.mean().plot(
    ax=ax, linewidth=0, marker="*", markersize=30, color="red", label="Mean Top 10",
)
df_worst.mean().plot(
    ax=ax, linewidth=0, marker="o", markersize=25, color="blue", label="Mean Bottom 10",
)
ax.legend(fontsize=18)
ax.set(
    ylabel="Return %",
    title="Mean Top-10 / Bottom-10 Portfolio Returns\n(avg of 10 runs per model)",
)
ax.grid(True)
fig.tight_layout()

# Try a hypothetical company and see the prediction

In [ ]:
print(f"Mean feature values:\n{X.mean().round(3)}\n\nMean return: {y.mean():.4f}")

In [ ]:
def predict_hypothetical(model, overrides: dict | None = None) -> float:
    """Create an 'average' company, apply overrides, and predict return."""
    company = pd.DataFrame(X.mean().values.reshape(1, -1), columns=X.columns)
    if overrides:
        for col, val in overrides.items():
            company[col] = val
    pred = model.predict(company)[0]
    print(f"Predicted return: {pred:.4f}")
    return pred

In [ ]:
# Average company
predict_hypothetical(rf_regressor)

In [ ]:
# "Good" company — high earnings relative to price
predict_hypothetical(rf_regressor, {
    "EV/EBIT": 5, "Op. In./(NWC+FA)": 0.4, "P/E": 3, "P/B": 4, "P/S": 13, "EBIT/TA": 1,
})

In [ ]:
# "Bad" company — overvalued on most metrics
predict_hypothetical(rf_regressor, {
    "EV/EBIT": 900, "Op. In./(NWC+FA)": -0.5, "P/E": 30, "P/B": 300, "P/S": 400, "EBIT/TA": 0.04,
})